In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

API_URL = "https://www.omdbapi.com/"
TIMEOUT = 10
DEBUG = False


def get_api_key() -> str:
    """Читает ключ OMDb из файла .env."""
    load_dotenv(override=True)
    api_key = os.getenv("OMDB_API_KEY")
    if not api_key:
        raise ValueError("Ключ OMDB_API_KEY не найден в .env")
    return api_key


def fetch_movie(title: str, api_key: str) -> dict:
    """Отправляет GET-запрос к OMDb API и возвращает JSON как словарь."""
    response = requests.get(
        API_URL,
        params={"t": title, "apikey": api_key},
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def print_movie(movie: dict) -> None:
    """Выводит нужные поля фильма."""
    print(
        f"Фильм '{movie.get('Title')}' ({movie.get('Year')}) — "
        f"жанр: {movie.get('Genre')}. Режиссёр: {movie.get('Director')}."
    )
    print(f"Актёры: {movie.get('Actors')}")
    print(f"Рейтинг IMDb: {movie.get('imdbRating')}")


def main() -> None:
    try:
        api_key = get_api_key()
    except ValueError as error:
        print("Ошибка конфигурации:", error)
        return

    title = input("Введите фильм для поиска: ").strip()
    if not title:
        print("Название фильма не может быть пустым.")
        return

    try:
        movie = fetch_movie(title, api_key)
    except requests.exceptions.HTTPError as error:
        print("Ошибка HTTP:", error)
        return
    except requests.exceptions.RequestException as error:
        print("Не удалось выполнить запрос:", error)
        return

    if DEBUG:
        print(json.dumps(movie, ensure_ascii=False, indent=2))

    if movie.get("Response") == "False":
        print("Фильм не найден:", movie.get("Error"))
        return

    print_movie(movie)


if __name__ == "__main__":
    main()